# 作业 2：列表、字典和类

姓名：欧阳洋　　学号：20244080124

In [1]:
import hashlib
import os
import platform
import uuid

STUDENT_ID = 20244080124
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
print(f"[Python 版本] {platform.python_version()}")


[Conda 环境] python_work1
[学号]       20244080124
[机器指纹] 66ab728912afd30e
[RUN_ID]   20244080124-66ab728912afd30e
[Python 版本] 3.12.15


## 环境与实现约定

使用独立 Conda 环境 `python_work1`，从新内核依次运行。以下四部分对应题目的 14 项任务，
分别给出函数或类、样例、关键结果解释和异常验证。
向量和矩阵的数值元素采用 Python 的 int 或 float，本实现不把 bool 当作数值元素；维度必须是非负整数。

## 一、字符串的乐趣（3 项）

回文判断先忽略空格和大小写，再与倒序字符串比较；字母序判断在去空格和统一大小写后检查相邻字符是否非降序；
去元音函数保留不在 a/e/i/o/u 中的字符，并能处理全是元音的输入。
题目允许回文输入为英文字母、数字与空格，因此不额外假定需要处理任意标点。

In [2]:
def is_palindrome(s):
    s = s.replace(" ", "").lower()
    return s == s[::-1]


def is_abecedarian(s):
    s = s.replace(" ", "").lower()
    for i in range(len(s) - 1):
        if s[i] > s[i + 1]:
            return False
    return True


def remove_vowels(s):
    result = ""
    for ch in s:
        if ch.lower() not in "aeiou":
            result += ch
    return result



for text in ["level", "T A C O cat", "rats live on no evil star", "python", "123 21"]:
    print(f"is_palindrome({text!r}) = {is_palindrome(text)}")
for text in ["adder", "beet", "dog", "cat", "abcd efgh xyz"]:
    print(f"is_abecedarian({text!r}) = {is_abecedarian(text)}")
for text in ["cat", "audio", "AEIOU", "banana"]:
    print(f"remove_vowels({text!r}) = {remove_vowels(text)!r}")


is_palindrome('level') = True
is_palindrome('T A C O cat') = True
is_palindrome('rats live on no evil star') = True
is_palindrome('python') = False
is_palindrome('123 21') = True
is_abecedarian('adder') = True
is_abecedarian('beet') = True
is_abecedarian('dog') = False
is_abecedarian('cat') = False
is_abecedarian('abcd efgh xyz') = True
remove_vowels('cat') = 'ct'
remove_vowels('audio') = 'd'
remove_vowels('AEIOU') = ''
remove_vowels('banana') = 'bnn'


**结果分析：** `T A C O cat` 归一化后是 `tacocat`，因此为回文；`123 21` 去空格后也是回文。
`adder` 和 `beet` 允许重复字母，仍满足非降序；`dog`、`cat` 不满足。
`audio` 去掉元音后为 `d`，全部大写元音 `AEIOU` 的结果为空串，不应漏删相邻元音。

In [3]:
assert is_palindrome("") and is_palindrome("   ") and is_palindrome("12321")
assert not is_palindrome("1232")
assert is_abecedarian("") and is_abecedarian("a a b") and not is_abecedarian("ba")
assert remove_vowels("aeiouAEIOU") == "" and remove_vowels("") == ""
print("字符串边界验证通过：空串、空格、数字回文、重复字母、连续元音。")

exception_checks = []
def expect_error(label, error_type, operation):
    """执行指定操作，确认它抛出题目要求的异常，并显示原因。"""
    try:
        operation()
    except error_type as error:
        exception_checks.append(label)
        print(f"{label}：{type(error).__name__}，{error}")
    else:
        raise AssertionError(f"{label} 应抛出 {error_type.__name__}")


字符串边界验证通过：空串、空格、数字回文、重复字母、连续元音。


## 二、列表的乐趣（2 项）

list_reverse 检查参数类型并返回倒序列表；is_sorted 只沿序列前进一次，检查 n−1 对相邻元素，
在首次逆序时直接返回 False，不使用排序或反复扫描。空序列和单元素序列没有逆序，返回 True。

In [4]:
def list_reverse(a):
    if not isinstance(a, list):
        raise TypeError("输入必须是列表")
    return a[::-1]


def is_sorted(seq):
    for i in range(len(seq) - 1):
        if seq[i] > seq[i + 1]:
            return False
    return True



original_list = [1, 2, 3]
print("反转结果：", list_reverse(original_list), "原列表：", original_list)
for seq in [[1, 2, 2, 5], [1, 4, 3, 5], (1, 2, 3), "abcd", [], [5]]:
    print(f"is_sorted({seq!r}) = {is_sorted(seq)}")


反转结果： [3, 2, 1] 原列表： [1, 2, 3]
is_sorted([1, 2, 2, 5]) = True
is_sorted([1, 4, 3, 5]) = False
is_sorted((1, 2, 3)) = True
is_sorted('abcd') = True
is_sorted([]) = True
is_sorted([5]) = True


In [5]:
assert list_reverse([]) == [] and list_reverse([1, 2, 3]) == [3, 2, 1]
assert original_list == [1, 2, 3]
assert is_sorted(()) and is_sorted("") and is_sorted(range(4))
assert is_sorted([1, 1]) and not is_sorted((2, 1))
expect_error("非列表输入", TypeError, lambda: list_reverse((1, 2, 3)))
expect_error("字符串不是列表", TypeError, lambda: list_reverse("abc"))
print("列表与一次遍历判断验证通过。")


非列表输入：TypeError，输入必须是列表
字符串不是列表：TypeError，输入必须是列表
列表与一次遍历判断验证通过。


**结果分析：** `[1,2,3]` 返回 `[3,2,1]`，原列表不变；非列表输入抛出 TypeError。
升序在本作业中允许相邻元素相等，因此 `[1,2,2,5]` 为 True。相邻比较次数至多 n−1，
对常用的列表、元组或字符串，该实现时间复杂度为 O(n)，额外空间为 O(1)。

## 三、作为向量的元组（4 项）

1. 元组乘标量返回同维元组，整数标量和整数元素有效。
2. 两向量内积为对应乘积之和，并显式返回 float；类型不符抛出 TypeError，长度不同抛出 ValueError。
3. 有效矩阵必须是元组的元组，每行等长且元素均为数值。
4. m×n 矩阵乘 n 维向量，逐行求内积，返回 m 维元组；调用 check_valid_mx 检查矩阵。

空矩阵 `()` 在这里约定为 0×0，可与空向量相乘；`((),())` 是 2×0 矩阵，与空向量相乘得到两个 0。

In [6]:
def is_number(x):
    return type(x) in (int, float)


def vec_scalar_mult(t, s):
    if type(t) is not tuple:
        raise TypeError("t必须是元组")
    if not is_number(s):
        raise TypeError("s必须是整数或浮点数")
    for x in t:
        if not is_number(x):
            raise TypeError("元组中的元素必须是整数或浮点数")
    return tuple(x * s for x in t)


def vec_inner_product(t1, t2):
    if type(t1) is not tuple or type(t2) is not tuple:
        raise TypeError("两个输入都必须是元组")
    if len(t1) != len(t2):
        raise ValueError("两个元组长度不一致")
    for x in t1 + t2:
        if not is_number(x):
            raise TypeError("元组中的元素必须是整数或浮点数")
    total = 0
    for i in range(len(t1)):
        total += t1[i] * t2[i]
    return float(total)


def check_valid_mx(mx):
    if type(mx) is not tuple:
        return False
    if len(mx) == 0:
        return True
    for row in mx:
        if type(row) is not tuple:
            return False
        if len(row) != len(mx[0]):
            return False
        for x in row:
            if not is_number(x):
                return False
    return True


def mx_vec_mult(mx, v):
    if not check_valid_mx(mx):
        raise TypeError("输入的矩阵不合法")
    if type(v) is not tuple:
        raise TypeError("向量必须是元组")
    for x in v:
        if not is_number(x):
            raise TypeError("向量元素必须是整数或浮点数")
    column_num = len(mx[0]) if len(mx) > 0 else 0
    if column_num != len(v):
        raise ValueError("矩阵列数和向量长度不一致")
    answer = []
    for row in mx:
        answer.append(vec_inner_product(row, v))
    return tuple(answer)



print("整数标量乘法：", vec_scalar_mult((1, 2, 3), 2))
print("浮点标量乘法：", vec_scalar_mult((1, 2.0, 3), 0.5))
print("内积及其类型：", vec_inner_product((1, 2, 3), (4, 5, 6)), type(vec_inner_product((1, 2, 3), (4, 5, 6))).__name__)
mx = ((1, 2, 3), (4, 5, 6))
print("矩阵有效：", check_valid_mx(mx))
print("不等长行：", check_valid_mx(((1, 2), (3,))))
print("矩阵乘向量：", mx_vec_mult(mx, (1, 0, -1)))


整数标量乘法： (2, 4, 6)
浮点标量乘法： (0.5, 1.0, 1.5)
内积及其类型： 32.0 float
矩阵有效： True
不等长行： False
矩阵乘向量： (-2.0, -2.0)


In [7]:
assert vec_scalar_mult((1, 2, 3), 2) == (2, 4, 6)
assert vec_scalar_mult((1, 2.0), 0.5) == (0.5, 1.0) and vec_scalar_mult((), 2) == ()
assert vec_inner_product((1, 2, 3), (4, 5, 6)) == 32.0
assert type(vec_inner_product((), ())) is float
assert check_valid_mx(mx) and check_valid_mx(()) and check_valid_mx(((), ()))
for invalid in [[(1, 2)], ([1, 2],), ((1, 2), (3,)), ((1, "a"),)]:
    assert check_valid_mx(invalid) is False
assert mx_vec_mult(mx, (1, 0, -1)) == (-2.0, -2.0)
assert mx_vec_mult((), ()) == () and mx_vec_mult(((), ()), ()) == (0.0, 0.0)
expect_error("标量乘法的向量非元组", TypeError, lambda: vec_scalar_mult([1, 2], 2))
expect_error("标量乘法的标量非数值", TypeError, lambda: vec_scalar_mult((1, 2), "2"))
expect_error("向量元素非数值", TypeError, lambda: vec_scalar_mult((1, "x"), 2))
expect_error("内积输入非元组", TypeError, lambda: vec_inner_product([1, 2], (3, 4)))
expect_error("内积长度不一致", ValueError, lambda: vec_inner_product((1,), (1, 2)))
expect_error("内积元素非数值", TypeError, lambda: vec_inner_product(("x",), (1,)))
expect_error("矩阵行长度不一致", TypeError, lambda: mx_vec_mult(((1, 2), (3,)), (1, 2)))
expect_error("矩阵乘法向量非元组", TypeError, lambda: mx_vec_mult(mx, [1, 0, -1]))
expect_error("矩阵列数与向量维度不一致", ValueError, lambda: mx_vec_mult(mx, (1, 2)))
print("元组向量、矩阵和异常分支验证通过。")


标量乘法的向量非元组：TypeError，t必须是元组
标量乘法的标量非数值：TypeError，s必须是整数或浮点数
向量元素非数值：TypeError，元组中的元素必须是整数或浮点数
内积输入非元组：TypeError，两个输入都必须是元组
内积长度不一致：ValueError，两个元组长度不一致
内积元素非数值：TypeError，元组中的元素必须是整数或浮点数
矩阵行长度不一致：TypeError，输入的矩阵不合法
矩阵乘法向量非元组：TypeError，向量必须是元组
矩阵列数与向量维度不一致：ValueError，矩阵列数和向量长度不一致
元组向量、矩阵和异常分支验证通过。


**结果分析：** `(1,2,3)·(4,5,6)=4+10+18=32.0`，结果类型为 float。
示例矩阵两行与 `(1,0,−1)` 的内积分别为 `1−3=−2`、`4−6=−2`，得到 `(-2.0,-2.0)`。
不等长行会使矩阵无效；矩阵列数与向量长度不一致时不能相乘。以上异常是有意测试并被捕获，
不应让 Notebook 在这些演示处中断。

## 四、更多向量的乐趣（5 项）

Vector 的 dimension 和 elements 必须是**实例属性**：每个对象可以有自己的维度和坐标，
若共用可变的类属性，可能让一个对象的修改影响其他对象。构造时将元素复制为元组；未提供元素时创建对应维度的零向量。

比较采用 Python 元组的字典序，先比较首坐标，只有相等才继续比较后面的坐标，不是按向量长度或模长排序。
dot 返回内积数值；标量乘法和两个向量的乘法都返回新的 Vector，其中 v*w 是逐元素乘法，不是内积。

In [8]:
class Vector:
    def __init__(self, dimension, elements=None):
        if type(dimension) is not int:
            raise TypeError("维度必须是整数")
        if dimension < 0:
            raise ValueError("维度不能是负数")
        if elements is None:
            elements = [0] * dimension
        if type(elements) not in (list, tuple):
            raise TypeError("元素必须使用列表或元组")
        if len(elements) != dimension:
            raise ValueError("维度和元素个数不一致")
        for x in elements:
            if not is_number(x):
                raise TypeError("每个元素都必须是整数或浮点数")
        self.dimension = dimension
        self.elements = tuple(elements)

    def check_other(self, other):
        if not isinstance(other, Vector):
            raise TypeError("另一个对象必须是Vector")
        if self.dimension != other.dimension:
            raise ValueError("两个向量维度不一致")

    def __eq__(self, other):
        self.check_other(other)
        return self.elements == other.elements

    def __lt__(self, other):
        self.check_other(other)
        return self.elements < other.elements

    def __le__(self, other):
        self.check_other(other)
        return self.elements <= other.elements

    def __gt__(self, other):
        self.check_other(other)
        return self.elements > other.elements

    def __ge__(self, other):
        self.check_other(other)
        return self.elements >= other.elements

    def dot(self, other):
        if not isinstance(other, Vector):
            raise TypeError("dot的参数必须是Vector")
        if self.dimension != other.dimension:
            raise ValueError("两个向量维度不一致")
        total = 0
        for i in range(self.dimension):
            total += self.elements[i] * other.elements[i]
        return float(total)

    def __mul__(self, other):
        if is_number(other):
            result = [x * other for x in self.elements]
            return Vector(self.dimension, result)
        if isinstance(other, Vector):
            if self.dimension != other.dimension:
                raise ValueError("两个向量维度不一致")
            result = []
            for i in range(self.dimension):
                result.append(self.elements[i] * other.elements[i])
            return Vector(self.dimension, result)
        raise TypeError("只能和标量或Vector相乘")

    def __rmul__(self, other):
        return self * other


v = Vector(2, (2, 4))
w = Vector(2, (2, 5))
print("v的元素：", v.elements)
print("w的元素：", w.elements)
print("v==w：", v == w)
print("v<w：", v < w)
print("v<=w：", v <= w)
print("w>v：", w > v)
print("v和w的内积：", v.dot(w))
print("2*v：", (2 * v).elements)
print("v*2：", (v * 2).elements)
print("v*w：", (v * w).elements)
print("三维零向量：", Vector(3).elements)
# 比较时直接利用元组本身的字典序。标量乘法和两个向量逐元素相乘都会返回新的 `Vector`，原来的向量不会改变。


v的元素： (2, 4)
w的元素： (2, 5)
v==w： False
v<w： True
v<=w： True
w>v： True
v和w的内积： 24.0
2*v： (4, 8)
v*2： (4, 8)
v*w： (4, 20)
三维零向量： (0, 0, 0)


In [9]:
assert Vector(0).elements == () and Vector(3).elements == (0, 0, 0)
assert Vector(2, [1, 2.0]).elements == (1, 2.0)
assert v == Vector(2, (2, 4)) and v != w and v < w and v <= w and w > v and w >= v
assert v.dot(w) == 24.0
assert (2 * v).elements == (v * 2).elements == (4, 8)
assert (0.5 * v).elements == (1.0, 2.0) and (v * w).elements == (4, 20)
a = Vector(2, (2, 3))
b = Vector(2, (4, 5))
product = a * b
print("题目示例逐元素乘法：", product.elements, "内积：", a.dot(b))
assert product.elements == (8, 15) and a.dot(b) == 23.0
assert product is not a and product is not b
assert a.elements == (2, 3) and b.elements == (4, 5)
source_elements = [1, 2]
copied_vector = Vector(2, source_elements)
source_elements[0] = 99
assert copied_vector.elements == (1, 2)
assert Vector(0).dot(Vector(0)) == 0.0
expect_error("维度不是整数", TypeError, lambda: Vector(2.0))
expect_error("维度为负数", ValueError, lambda: Vector(-1))
expect_error("维度与元素数不一致", ValueError, lambda: Vector(2, (1,)))
expect_error("元素容器类型错误", TypeError, lambda: Vector(2, "12"))
expect_error("元素不是数值", TypeError, lambda: Vector(2, (1, "x")))
expect_error("相等比较维度不一致", ValueError, lambda: v == Vector(3))
expect_error("排序比较维度不一致", ValueError, lambda: v < Vector(3))
expect_error("dot 参数类型不正确", TypeError, lambda: v.dot((1, 2)))
expect_error("dot 维度不一致", ValueError, lambda: v.dot(Vector(3)))
expect_error("逐元素乘法维度不一致", ValueError, lambda: v * Vector(3))
expect_error("乘法对象类型不正确", TypeError, lambda: v * "2")
print("Vector 初始化、六种比较、dot、左右标量乘法和逐元素乘法验证通过。")


题目示例逐元素乘法： (8, 15) 内积： 23.0
维度不是整数：TypeError，维度必须是整数
维度为负数：ValueError，维度不能是负数
维度与元素数不一致：ValueError，维度和元素个数不一致
元素容器类型错误：TypeError，元素必须使用列表或元组
元素不是数值：TypeError，每个元素都必须是整数或浮点数
相等比较维度不一致：ValueError，两个向量维度不一致
排序比较维度不一致：ValueError，两个向量维度不一致
dot 参数类型不正确：TypeError，dot的参数必须是Vector
dot 维度不一致：ValueError，两个向量维度不一致
逐元素乘法维度不一致：ValueError，两个向量维度不一致
乘法对象类型不正确：TypeError，只能和标量或Vector相乘
Vector 初始化、六种比较、dot、左右标量乘法和逐元素乘法验证通过。


**结果分析：** `(2,4)<(2,5)` 是因为首坐标相等、第二坐标 4<5。`!=` 依据 `__eq__` 的结果得到，
其余比较方法直接使用元组的对应比较。维度不同的向量在比较、dot 或逐元素乘法前都会抛出 ValueError。

对于题目示例，`(2,3)*(4,5)` 得到 Vector 的坐标 `(8,15)`，内积则是单个数 `23.0`；
左右标量乘法都返回新对象，原向量保持不变。实例接收列表时复制为元组，外部列表之后修改不会改变该向量的坐标。

In [10]:
assert STUDENT_ID == 20244080124
assert CONDA_ENV not in ("base", "") and os.path.isdir(os.path.join(os.environ.get("CONDA_PREFIX", ""), "conda-meta"))
assert v.dimension == 2 and Vector(3).dimension == 3
assert v.elements == (2, 4) and w.elements == (2, 5)
print(f"作业复核通过：14 项任务及 {len(exception_checks)} 个异常演示，身份与独立 Conda 环境正确。")


作业复核通过：14 项任务及 22 个异常演示，身份与独立 Conda 环境正确。


## 提交文件

保存并提交本 Notebook：`20244080124_欧阳洋_hw02.ipynb`。输出来自新内核顺序执行；字符串、列表、元组和类的函数均包含在文件内，不依赖外部自定义模块。